<a href="https://colab.research.google.com/github/amrzhd/EEG-MSCNN/blob/main/NLP_Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#NLP Audit

#Installing Packages

In [ ]:
!pip install torch-summary

In [ ]:
!pip install hazm

#Libraries Used

In [ ]:
import random
import numpy as np
import pandas as pd

import seaborn as sns
from tqdm import tqdm
from hazm import Normalizer

# Torch
import torch
from torchsummary import summary
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, BertForSequenceClassification, AdamW

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

#Data

In [ ]:
data = {
    'تاسیس شرکت': 1, 'تاسیس موسسه': 1, 'تاسیس موسسه ': 1, 'تاسیس شرکت و آورده نقدی سرمایه': 1,
    'درآمد نقدی': 2, 'درآمد نقدی طی فاکتور ': 2, 'ارائه خدمات به مشتری به صورت درآمد نقدی': 2,
    'فروش نقدی': 3, 'فروش نقدی طی فاکتور  به آقای': 3, 'ارائه کالا به مشتری به صورت فروش نقدی': 3,
    'فروش نسیه': 5, 'فروش نسیه طی فاکتور  به آقای': 5, 'فروش کالا به مشتری': 5,
    'فروش کالا اقساطی': 6, 'فروش کالا به صورت اقساطی': 6,
    'برداشت آقای': 7, 'برداشت سهامدار': 7, 'برداشت سهامداران': 7, 'برداشت شریک': 7, 'برداشت شرکا': 7,
    'هزینه حمل طبق بارنامه  به صورت نقد': 8, 'هزینه حمل طبق بارنامه  به صورت نسیه': 9,
    'پرداخت قرض به آقای': 10, 'دریافت وجه فاکتور نسیه  از آقای': 11,
    'خرید یک دستگاه کامپیوتر ف ': 12,
    'جابجایی وجه از صندوق به بانک': 13, 'جابجایی وجه از بانک به صندوق': 14,
    'انتقال وجه بین حساب های بانکی': 15, 'انتقال وجه میان حساب های بانکی': 15, 'جابجایی بانکی': 15, 'جابجایی بانکی از حساب  به حساب ': 15,
    'دریافت چک  طی فاکتور فروش ': 16, 'وصول چک  ': 17,
    'برداشت چک  توسط جاری شرکا': 18, 'پرداخت چک  بابت خرید نسیه فاکتور ': 19,
    'وصول چک ': 20, 'تامین نقدی چک  توسط جاری شرکا': 21,
    'بستن ارزش افزوده فصل ': 22, 'پرداخت ارزش افزوده فصل ': 22,
    'افتتاحیه': 23, 'افتتاحیه دستی': 24, 'اصلاح افتتاحیه': 25,
    'کارمزد خدمات بانکی': 26, 'اصلاح سند اتوماتیک': 27,
    'اختتامیه': 28, 'اختتامیه دستی': 29,
    'جابجایی حساب': 30, 'جابه جایی حساب': 30,
    'هزینه اسنپ': 31, 'اسنپ آقای': 31, 'هزینه آژانس': 32, 'آژانس آقای': 32,
    'بستن حسابهای موقت': 33, 'هزینه پست': 34, 'سود سپرده بانکی': 35,
    'هزینه ناهار': 36, 'غذای آقای': 37, 'هزینه پیک': 38, 'شارژ تنخواه': 39, 'شارژ تنخواه آقای': 39,
    'پرداخت مساعده به آقای': 40,
    'تایید پیشنهاد قیمت در سامانه ستاد': 41,
    'پرداخت بخشی از حقوق ... ماه پرسنل': 42, 'پرداخت حقوق ماه 1 پرسنل': 42,
    'هزینه شیرینی': 43,
    'خرید مواد اولیه جهت تولید': 44, 'تهیه مواد اولیه مورد نیاز جهت تولید': 44, 'تهیه مواد اولیه برای تولید': 44,
    'پرداخت حقوق و دستمزد کارکنان': 45, 'واریز حقوق و دستمزد به کارکنان': 45,
    'دریافت واریز نقدی از مشتریان': 46, 'دریافت وجه نقد از مشتریان': 46, 'تسویه بخشی از طلب مشتری': 46,
    'پرداخت اجاره ماهانه محل کار': 47, 'پرداخت هزینه اجاره ماهانه محل کار': 47,
    'تسویه حساب با تأمین کنندگان': 48, 'تصفیه حساب های تأمین کنندگان': 48, 'تسویه حساب پرداختنی تجاری تامین کنندگان کالا': 48,
    'تسویه حساب آقای تامین کننده کالا': 48, 'تسویه حساب آقای تامین کننده مواد اولیه': 48,
    'پرداخت قبض آب': 49, 'پرداخت صورتحساب آب': 49, 'پرداخت صورتحساب آب ماه': 49,
    'پرداخت قبض برق': 50, 'پرداخت صورتحساب برق': 50, 'پرداخت صورتحساب برق ماه': 50,
    'پرداخت قبض گاز': 51, 'پرداخت صورتحساب گاز': 51, 'پرداخت صورتحساب گاز ماه': 51,
    'پرداخت قبض تلفن': 52, 'پرداخت صورتحساب تلفن': 52, 'پرداخت صورتحساب تلفن ماه': 53,
    'خرید تجهیزات اداری': 54, 'تهیه و خرید تجهیزات اداری': 54, 'تامین تجهیزات اداری': 54,
    'ارائه خدمات مشاوره و دریافت درآمد': 55, 'ارائه خدمات مشاوره ای و کسب درآمد': 55,
    'خرید نرم افزار و تجهیزات کامپیوتری': 56, 'تهیه نرم افزار و تجهیزات کامپیوتری': 56, 'تامین نرم افزار و تجهیزات کامپیوتری': 56,
    'پرداخت هزینه های تعمیر و نگهداری': 57, 'پرداخت هزینه های مربوط به تعمیر و نگهداری': 57,
    'دریافت درآمد اجاره از املاک': 58, 'دریافت وجه اجاره از املاک': 58,
    'پرداخت مالیات بر درآمد': 59, 'واریز مالیات بر درآمد به حساب سازمان امور مالیاتی': 59,
    'خرید کالا نقدی': 60, 'خرید کالا به صورت نقدی': 60,
    'خرید کالا نسیه': 61, 'خرید کالا به صورت نسیه': 61,
    'پرداخت بدهی های بانکی': 62, 'تسویه بدهی های بانکی': 62,
    'دریافت وام از بانک': 63, 'دریافت وام بانکی': 63, 'اخذ وام از بانک': 63, 'اخذ وام بانکی': 63,
    'دریافت چک از مشتریان': 64, 'دریافت چک های دریافتنی از مشتریان': 64, 'اخذ چک از مشتری': 64,
    'پرداخت کارمزد به واسطه های مالی': 65, 'پرداخت هزینه کارمزد به واسطه های مالی': 65,
    'خرید ارز': 66, 'معامله ارز (خرید)': 66, 'فروش ارز': 66, 'معامله ارز (فروش ارز)': 66,
    'پرداخت فاکتور به پیمانکار': 67, 'پرداخت فاکتور به پیمانکاران': 67, 'پرداخت صورتوضعیت به پیمانکار': 67, 'پرداخت صورتوضعیت به پیمانکاران': 67,
    'دریافت وجه بابت خدمات پس از فروش': 68, 'دریافت وجه بابت خدمات پس از فروش کالا': 68,
    'پرداخت هزینه تبلیغات و بازاریابی': 69, 'پرداخت هزینه های تبلیغاتی و بازاریابی': 69,
    'دریافت سپرده ضمانت از مشتریان': 70, 'دریافت مبلغ سپرده تضمینی از مشتریان': 70,
    'پرداخت هزینه مشاوره حقوقی': 71, 'پرداخت هزینه های مشاوره حقوقی': 71,
    'دریافت سرمایه گذاری جدید': 72, 'جذب سرمایه گذاری جدید': 72,
    'پرداخت هزینه بهداشتی و درمانی کارکنان': 73, 'پرداخت هزینه های بهداشتی و درمانی پرسنل': 73,
    'دریافت بهره از سرمایه گذاری های انجام شده': 74, 'دریافت سود سرمایه گذاری های انجام شده': 74,
    'قبض گاز': 75, 'قبض تلفن': 76, 'قبض آب': 77, 'قبض برق': 78,
    'شناسایی هزینه اجاره': 79, 'شناسایی درآمد اجاره': 80
}

# Print the dictionary
print("[")
for key, value in data.items():
    print(f"    '{key}': {value},")
print("}")


##Dataset Augmenter

In [ ]:
# Lists of contextually related words to be added as prefixes and suffixes.
prefixes = {
    'تاسیس': ['جدید', 'سریع', 'موفق', 'پیشرو', 'برتر'],
    'درآمد': ['سریع', 'پیشرو', 'معتبر', 'پرمخاطب', 'موفق'],
    'فروش': ['سریع', 'مشتری‌مدار', 'پر رونق', 'موفق', 'پیشرو'],
    'برداشت': ['سریع', 'معتبر', 'کارآمد', 'برتر', 'نوین'],
    'هزینه': ['حرفه‌ای', 'کارآمد', 'پیشرو', 'معتبر', 'متنوع'],
    'پرداخت': ['سریع', 'به موقع', 'مطمئن', 'موفق', 'پیشرو'],
    'دریافت': ['سریع', 'به موقع', 'معتبر', 'موفق', 'پیشرو'],
    'جابجایی': ['سریع', 'کارآمد', 'پیشرو', 'معتبر', 'موثر'],
    'انتقال': ['سریع', 'موثر', 'پیشرو', 'معتبر', 'موفق'],
    'وصول': ['سریع', 'به موقع', 'معتبر', 'موثر', 'پیشرو'],
    'افتتاحیه': ['برنامه‌ریزی‌شده', 'موفق', 'سریع', 'جدید', 'پیشرو'],
    'اختتامیه': ['محسوب', 'موفق', 'سریع', 'جدید', 'پیشرو'],
    'شناسایی': ['دقیق', 'سریع', 'کارآمد', 'معتبر', 'موثر']
}

suffixes = {
    'شرکت': ['موفق', 'جدید', 'پرمحتوا', 'برتر', 'نوین'],
    'موسسه': ['جدید', 'سریع', 'پیشرو', 'موفق', 'معتبر'],
    'نقدی': ['سریع', 'به موقع', 'مطمئن', 'موفق', 'موثر'],
    'فاکتور': ['به موقع', 'معتبر', 'سریع', 'کارآمد', 'پیشرو'],
    'آقای': ['موفق', 'پرمحتوا', 'پیشرو', 'جدید', 'معتبر'],
    'کالا': ['سریع', 'به موقع', 'کارآمد', 'موفق', 'پیشرو'],
    'بانکی': ['سریع', 'به موقع', 'کارآمد', 'موثر', 'پیشرو']
}

def augment_text(text):
    # Choose a prefix based on a key word match.
    chosen_prefix = ''
    for key, words in prefixes.items():
        if key in text:
            chosen_prefix = random.choice(words)
            break
    # Choose a suffix based on a key word match.
    chosen_suffix = ''
    for key, words in suffixes.items():
        if key in text:
            chosen_suffix = random.choice(words)
            break
    # Randomly decide to add prefix, suffix, both, or none.
    option = random.choice(['prefix', 'suffix', 'both', 'none'])
    if option == 'prefix' and chosen_prefix:
        return chosen_prefix + " " + text
    elif option == 'suffix' and chosen_suffix:
        return text + " " + chosen_suffix
    elif option == 'both' and chosen_prefix and chosen_suffix:
        return chosen_prefix + " " + text + " " + chosen_suffix
    else:
        return text

# Generate augmented samples until we have 10,000 rows.
augmented_samples = []
base_texts = list(data.keys())
target = 10000
while len(augmented_samples) < target:
    text = random.choice(base_texts)
    label = data[text]
    new_text = augment_text(text)
    augmented_samples.append((new_text, label))

# Write the augmented samples to a CSV file.
with open("augmented_data.csv", "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["text", "label"])
    for row in augmented_samples:
        writer.writerow(row)

print("CSV file 'augmented_data.csv' generated with", len(augmented_samples), "samples.")


##Persian Text Normalizer

In [ ]:
class PersianTextNormalizer():
    def __init__(self, file_name):
        self.file_name = file_name
        self.df = pd.read_csv(self.file_name)

        self.arabic_persian_mapping = {
            chr(65153): chr(1570), chr(65154): chr(1570),
            chr(65165): chr(1575), chr(65166): chr(1575),
            chr(65156): chr(1571), chr(65167): chr(1576),
            chr(65168): chr(1576), chr(65169): chr(1576),
            chr(65170): chr(1576), chr(64342): chr(1662),
            chr(64343): chr(1662), chr(64344): chr(1662),
            chr(64345): chr(1662), chr(65173): chr(1578),
            chr(65174): chr(1578), chr(65175): chr(1578),
            chr(65176): chr(1578), chr(65177): chr(1579),
            chr(65178): chr(1579), chr(65179): chr(1579),
            chr(65180): chr(1579), chr(65275): chr(1604) + chr(1575),
            chr(65276): chr(1604) + chr(1575),
            chr(65010): chr(1575) + chr(1604) + chr(1604) + chr(1607)
        }

        self.arabic_number_mapping = {
            chr(1776): '0', chr(1777): '1', chr(1778): '2',
            chr(1779): '3', chr(1780): '4', chr(1781): '5',
            chr(1782): '6', chr(1783): '7', chr(1784): '8', chr(1785): '9'
        }

        self.redundant_char_list = [
                chr(1958), chr(59424), chr(1960), chr(59430), chr(1962),
                chr(59425), chr(1959), chr(1961), chr(1963), chr(59429),
                chr(1620), chr(173), chr(8301), chr(1600), chr(8205),
                chr(8207), chr(31), chr(8235), chr(8236), chr(65136),
                chr(8204), chr(32) + chr(46), chr(46), chr(8216), chr(32) + chr(1548),
                chr(1548), chr(32) + chr(1563), chr(1563), chr(45), chr(1475),
                chr(65109), chr(65306), chr(32) + chr(58), chr(58), chr(8230),
                chr(46) + chr(32) + chr(46) + chr(32) + chr(46), chr(65281), chr(32) + chr(33), chr(33), chr(65311),
                chr(32) + chr(1567), chr(63), chr(1567), chr(40), chr(41),
                chr(1607) + chr(1569), chr(1574) + chr(1574), chr(1574) + chr(1740), chr(172), chr(160),
                chr(8203),
                '\t', "^31", "^91", "^93", "^11",
                "^13", "^m",  chr(32) + "^p", "^p" + chr(32), "^p" * 4,
                "^p" * 3
            ]

    def persian_alphabetic_standardizer(self):
        """Replace Arabic characters with Persian equivalents in the dataframe."""
        self.df = self.df.applymap(lambda x: self.replace_chars(x, self.arabic_persian_mapping) if isinstance(x, str) else x)

    def arabic_number_corrector(self):
        """Replace Arabic numbers with Persian numbers in the dataframe."""
        self.df = self.df.applymap(lambda x: self.replace_chars(x, self.arabic_number_mapping) if isinstance(x, str) else x)

    def replace_chars(self, text, mapping):
        for old_char, new_char in mapping.items():
            text = text.replace(old_char, new_char)
        return text

    def remove_redundant(self):
    """Remove redundant characters from the dataframe using redundant_char_list."""
      def remove_chars(text):
          for redundant in self.redundant_char_list:
              text = text.replace(redundant, '')
          return text

      self.df = self.df.applymap(
          lambda x: remove_chars(x) if isinstance(x, str) else x
      )

## Dataset CLass

In [ ]:
class PersianTextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        """
        df (pd.DataFrame): Pre-normalized dataframe.
        tokenizer: HuggingFace tokenizer.
        max_length (int): Maximum sequence length.
        train (bool): If True, loads training set; else, loads test set.
        """
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.hazm_normalizer = Normalizer()
        self.df = df
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    def normalize_text(self, text):
        return self.hazm_normalizer.normalize(text)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        normalized_text = self.normalize_text(row['text'])

        encoding = self.tokenizer.encode_plus(
            normalized_text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        encoding = {key: value.squeeze() for key, value in encoding.items()}
        encoding['labels'] = torch.tensor(row['labels'], dtype=torch.long)
        return encoding


##Building Dataset

In [ ]:
# Load the dataset from Excel file
file_name = "Dataset.csv"
normalizer = PersianTextNormalizer(file_name)
normalizer.persian_alphabetic_standardizer()
normalizer.arabic_number_corrector()
df = normalizer.df

# Convert categorical labels to numerical if needed **before splitting**
if df['labels'].dtype == object:
    label_mapping = {label: idx for idx, label in enumerate(df['labels'].unique())}
    df['labels'] = df['labels'].map(label_mapping)

df['labels'] = df['labels'] - 1
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("HooshvareLab/bert-fa-base-uncased")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Split data into train and test sets (80% train, 20% test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['labels'])

# Create dataset instances for training and testing data
train_dataset = PersianTextDataset(train_df, tokenizer)
test_dataset = PersianTextDataset(test_df, tokenizer)


#Early Stopping

In [ ]:
class EarlyStopping():
    def __init__(self, patience=5, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_model = None
        self.best_loss = None
        self.counter = 0
        self.status = ""

    def __call__(self, model, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
        elif self.best_loss - val_loss >= self.min_delta:
            self.best_model = copy.deepcopy(model.state_dict())
            self.best_loss = val_loss
            self.counter = 0
            self.status = f"Improvement found, counter reset to {self.counter}"
        else:
            self.counter += 1
            self.status = f"No improvement in the last {self.counter} epochs"
            if self.counter >= self.patience:
                self.status = f"Early stopping triggered after {self.counter} epochs."
                if self.restore_best_weights:
                    model.load_state_dict(self.best_model)
                return True
        return False


#Training Class

In [ ]:
class TrainModel():
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def train_model(self, model, train_dataset, epochs=3, batch_size=32, lr=2e-5):
        model = model.to(self.device)
        optimizer = AdamW(model.parameters(), lr=lr)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        highest_train_accuracy = 0.0

        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            correct = 0
            total = 0

            for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                optimizer.zero_grad()
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                running_loss += loss.item()
                loss.backward()
                optimizer.step()

                # Compute accuracy
                preds = torch.argmax(outputs.logits, dim=-1)
                total += labels.numel()
                correct += (preds == labels).sum().item()

            epoch_loss = running_loss / len(train_loader)
            epoch_accuracy = correct / total
            if epoch_accuracy > highest_train_accuracy:
                highest_train_accuracy = epoch_accuracy

            print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}, Accuracy: {(epoch_accuracy*100):.2f}%")

        average_loss = running_loss / len(train_loader.dataset)
        print("Average Loss:", average_loss)

        # Saving model
        torch.save({'model_state_dict': model.state_dict()}, 'finetuned_nlp_model.pth')
        return model

    def train_early_stopping_model(self, model, train_dataset, valid_dataset, epochs=3, batch_size=32, lr=2e-5, patience=5, min_delta=0):
        early_stopping = EarlyStopping(patience=patience, min_delta=min_delta, restore_best_weights=True)
        model = model.to(self.device)
        optimizer = AdamW(model.parameters(), lr=lr)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

        highest_train_accuracy = 0.0

        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            correct = 0
            total = 0

            # Training phase
            for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                optimizer.zero_grad()
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                running_loss += loss.item()
                loss.backward()
                optimizer.step()

                # Compute training accuracy
                preds = torch.argmax(outputs.logits, dim=-1)
                total += labels.numel()
                correct += (preds == labels).sum().item()

            train_loss = running_loss / len(train_loader)
            train_accuracy = correct / total

            # Validation phase
            model.eval()
            val_loss = 0.0
            val_total = 0
            with torch.no_grad():
                for batch in valid_loader:
                    input_ids = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    labels = batch['labels'].to(self.device)

                    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                    loss = outputs.loss
                    # Multiply loss by batch size to accumulate total loss
                    val_loss += loss.item() * labels.size(0)
                    val_total += labels.size(0)
            avg_val_loss = val_loss / val_total

            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {(train_accuracy*100):.2f}%, Val Loss: {avg_val_loss:.4f}")

            # Early stopping check
            if early_stopping(model, avg_val_loss):
                print(early_stopping.status)
                break

        # Save final model weights
        torch.save({'model_state_dict': model.state_dict()}, 'finetuned_nlp_model_early_stop.pth')
        return model


#Evaluating Class

In [ ]:
class EvalModel():
    def __init__(self, model, test_dataset):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    def test_model(self):
        self.model.eval()
        total_loss = 0
        correct_predictions = 0
        total_examples = 0

        with torch.no_grad():
            for batch in tqdm(self.test_loader, desc="Evaluating"):
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                logits = outputs.logits

                total_loss += loss.item()
                predictions = torch.argmax(logits, dim=1)
                correct_predictions += torch.sum(predictions == labels).item()
                total_examples += labels.size(0)

        avg_loss = total_loss / len(self.test_loader)
        accuracy = correct_predictions / total_examples

        print("\n/------------------------------/")
        print(f"Test Accuracy: {accuracy * 100:.2f}%")
        print("/------------------------------/\n")

    def plot_confusion_matrix(self):
        self.model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in tqdm(self.test_loader, desc="Generating Confusion Matrix"):
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                predictions = torch.argmax(outputs.logits, dim=1)

                all_preds.extend(predictions.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # Compute confusion matrix
        cm = confusion_matrix(all_labels, all_preds)

        # Plot confusion matrix
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=self.class_names, yticklabels=self.class_names)
        plt.xlabel("Predicted Labels")
        plt.ylabel("True Labels")
        plt.title("Confusion Matrix")
        plt.show()


#ParsBert Model

In [ ]:
class ParsBERTClassifier(torch.nn.Module):
    def __init__(self, num_labels=80):
        super(ParsBERTClassifier, self).__init__()
        self.model = BertForSequenceClassification.from_pretrained(
            "HooshvareLab/bert-fa-base-uncased",  # ParsBERT pretrained model
            num_labels=num_labels
        )

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)


###Model Summary

In [ ]:
input_shape = (1, 128)
parsbert_model = ParsBERTClassifier().to(device)
input_ids = torch.zeros(input_shape, dtype=torch.long).to(device)
attention_mask = torch.zeros(input_shape, dtype=torch.long).to(device)
summary(parsbert_model, input_data=(input_ids, attention_mask))

###Training Model

In [ ]:
# Define the optimizer
LEARNING_RATE = 2e-5
EPOCHS = 3
NUM_LABELS = 81
BATCH_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
parsbert_model = ParsBERTClassifier(num_labels=NUM_LABELS)
trainer = TrainModel()

finetuned_nlp_model = trainer.train_model(
    parsbert_model,
    train_dataset,
    lr=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS
)

finetuned_nlp_model = trainer.train_early_stopping_model(
    parsbert_model,
    train_dataset,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    valid_dataset,  # assume you have a validation dataset instance
    patience=5,
    min_delta=0
)

###Evaluating Model

In [ ]:
NUM_LABELS = 80
parsbert_model = ParsBERTClassifier(num_labels=NUM_LABELS)
evaluator = EvalModel(parsbert_model, test_dataset)
# evaluator.test_model()
evaluator.plot_confusion_matrix()